# Build the Strack APK on Google Colab

A free, no-queue way to build an installable Android APK in the cloud. Run the cells **top to bottom**. Total time is ~10–20 min (the toolchain is reinstalled each Colab session).

- No signing secrets needed — Expo's generated Android project signs the release build with the debug keystore.
- Uses `npm install` (not `npm ci`) on purpose, so the Windows-generated lockfile works on Colab's Linux.
- The final cell downloads `app-release.apk` to your computer.

In [ ]:
# 1) JDK 17 + Node 20
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get -qq install -y nodejs
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!java -version && node -v && npm -v

In [ ]:
# 2) Android SDK (command-line tools + platform 36 + build-tools)
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O cmd.zip
!mkdir -p /root/android-sdk/cmdline-tools && unzip -q -o cmd.zip -d /root/android-sdk/cmdline-tools
!mv -f /root/android-sdk/cmdline-tools/cmdline-tools /root/android-sdk/cmdline-tools/latest 2>/dev/null || true
import os
os.environ['ANDROID_HOME'] = '/root/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/root/android-sdk'
os.environ['PATH'] += ':/root/android-sdk/cmdline-tools/latest/bin:/root/android-sdk/platform-tools'
!yes | sdkmanager --licenses >/dev/null
!sdkmanager 'platform-tools' 'platforms;android-36' 'build-tools;36.0.0' >/dev/null
print('Android SDK ready')

In [ ]:
# 3) Clone, install deps, prebuild, and build the release APK
%cd /content
!rm -rf strack
!git clone --depth 1 https://github.com/divinefavourak/strack.git
%cd /content/strack
!npm install --no-audit --no-fund
!npx expo prebuild --platform android --no-install
# Give Gradle more heap so the Hermes/bundling step doesn't get OOM-killed
!mkdir -p ~/.gradle && echo 'org.gradle.jvmargs=-Xmx5g' > ~/.gradle/gradle.properties
# arm64-v8a only keeps the APK small (covers essentially all modern phones).
# Drop the -P flag for a universal APK, or add ',armeabi-v7a' for older devices.
!cd android && chmod +x gradlew && ./gradlew assembleRelease -PreactNativeArchitectures=arm64-v8a --no-daemon

In [ ]:
# 4) Download the APK
import glob
from google.colab import files
apks = glob.glob('/content/strack/android/app/build/outputs/apk/release/*.apk')
assert apks, 'No APK found — check the build cell output above for errors.'
print('APK:', apks[0])
files.download(apks[0])